# JCJC ミニ四駆 エネルギー分析

JCJC（ジャパンカップジュニアサーキット）を走行するミニ四駆の  
**投入エネルギー（電池）** と **走行エネルギー** を定量的に整理します。

---

## 分析の流れ

1. コースレイアウトの読み込み・総距離の算出
2. 電池エネルギーの計算（投入エネルギー）
3. 走行エネルギーの計算（運動エネルギー＋位置エネルギー変化）
4. エネルギー効率の推定
5. セクション別タイム分析

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# 日本語フォント設定
matplotlib.rcParams['font.family'] = 'MS Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

print('ライブラリ読み込み完了')

## 1. コースレイアウトの読み込み

In [ ]:
course_df = pd.read_csv('../data/course.csv')
print(course_df.to_string(index=False))

In [ ]:
# コース総距離
total_length_mm = course_df['length_mm'].sum()
total_length_m  = total_length_mm / 1000

# スロープの高低差
slope_sections = course_df[course_df['type'] == 'slope_up']
height_gain_mm = (slope_sections['length_mm'] * np.sin(np.radians(slope_sections['slope_deg']))).sum()

slope_sections_down = course_df[course_df['type'] == 'slope_down']
height_loss_mm = (slope_sections_down['length_mm'] * np.sin(np.radians(slope_sections_down['slope_deg']))).sum()

print(f'コース1周の総距離 : {total_length_mm:.0f} mm  ({total_length_m:.3f} m)')
print(f'スロープ上り高さ  : {height_gain_mm:.1f} mm  ({height_gain_mm/1000*1000:.1f} mm)')
print(f'スロープ下り高さ  : {height_loss_mm:.1f} mm')
print(f'1周の正味高低差   : {height_gain_mm - height_loss_mm:.1f} mm')

## 2. 電池エネルギーの計算（投入エネルギー）

電池が供給できる最大エネルギー（理論値）:

$$E_{\text{battery}} = V_{\text{avg}} \times C \times 3600 \quad [\text{J}]$$

ここで $V_{\text{avg}} = (V_{\text{start}} + V_{\text{end}}) / 2$、$C$ は容量 [Ah]。

In [ ]:
runs_df = pd.read_csv('../data/runs.csv')

# 投入エネルギー計算（2本直列を想定: voltage は合計値）
runs_df['V_avg']         = (runs_df['voltage_start'] + runs_df['voltage_end']) / 2
runs_df['delta_V']       = runs_df['voltage_start'] - runs_df['voltage_end']
runs_df['capacity_Ah']   = runs_df['capacity_mAh'] / 1000

# 使用した容量の推定（電圧降下量比で按分）
# ※実測電流がない場合は全容量を上限とし、簡易的に電圧降下比で補正
MAX_DELTA_V = 0.6   # 2本で 0.6V 降下時に満容量使い切り（粗い近似）
runs_df['used_capacity_Ah'] = runs_df['capacity_Ah'] * (runs_df['delta_V'] / MAX_DELTA_V).clip(0, 1)

runs_df['E_battery_J']   = runs_df['V_avg'] * runs_df['used_capacity_Ah'] * 3600

print(runs_df[['date','battery_type','V_avg','delta_V','used_capacity_Ah','E_battery_J','laps','total_time_s']].to_string(index=False))

## 3. 走行エネルギーの計算

走行に必要な力学的エネルギー（概算）:

$$E_{\text{run}} = \frac{1}{2} m v_{\text{avg}}^2 + m g \Delta h \quad [\text{J}]$$

> - $m$：車重 [kg]  
> - $v_{\text{avg}}$：平均速度 [m/s]  
> - $\Delta h$：1周あたり正味高低差 [m]（JCJCは概ゼロ）  

**注意**：摩擦・空気抵抗・ギア損失は含まれないため、実際のモーター消費はこれより大きい。

In [ ]:
g = 9.80665  # [m/s²]

runs_df['car_weight_kg'] = runs_df['car_weight_g'] / 1000
runs_df['total_distance_m'] = runs_df['laps'] * total_length_m
runs_df['v_avg_m_s']     = runs_df['total_distance_m'] / runs_df['total_time_s']

# 位置エネルギー変化（JCJCは1周で元の高さに戻るので 0、ただし途中の上昇分を考慮する場合）
net_height_m = (height_gain_mm - height_loss_mm) / 1000  # 通常 ≈ 0

# 運動エネルギー（最高速度推定: 平均速度の ~1.4倍と仮定）
runs_df['v_max_est_m_s'] = runs_df['v_avg_m_s'] * 1.4
runs_df['E_kinetic_J']   = 0.5 * runs_df['car_weight_kg'] * runs_df['v_max_est_m_s'] ** 2

# 位置エネルギー（スロープ最高点、1周中の最大高さ）
top_height_m = height_gain_mm / 1000
runs_df['E_potential_J'] = runs_df['car_weight_kg'] * g * top_height_m

# 走行中の合計力学的エネルギー（上り時の最大値）
runs_df['E_mechanical_J'] = runs_df['E_kinetic_J'] + runs_df['E_potential_J']

print(runs_df[['date','v_avg_m_s','v_max_est_m_s','E_kinetic_J','E_potential_J','E_mechanical_J','E_battery_J']].to_string(index=False))

## 4. エネルギー効率の推定

$$\eta = \frac{E_{\text{mechanical}}}{E_{\text{battery}}} \times 100 \quad [\%]$$

残りのエネルギーはモーター・ギア・タイヤの発熱として失われています。

In [ ]:
runs_df['efficiency_pct'] = (runs_df['E_mechanical_J'] / runs_df['E_battery_J']) * 100

summary = runs_df[['date','battery_type','note','E_battery_J','E_mechanical_J','efficiency_pct']].copy()
summary.columns = ['日付','電池種類','備考','投入E[J]','力学E[J]','効率[%]']
print(summary.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('JCJC ミニ四駆 エネルギー分析サマリー', fontsize=14)

labels = runs_df['date'] + '\n' + runs_df['battery_type']

# --- 投入エネルギー vs 力学エネルギー ---
x = np.arange(len(runs_df))
w = 0.35
axes[0].bar(x - w/2, runs_df['E_battery_J'], w, label='投入エネルギー', color='steelblue')
axes[0].bar(x + w/2, runs_df['E_mechanical_J'], w, label='力学エネルギー', color='orange')
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, fontsize=8)
axes[0].set_ylabel('エネルギー [J]')
axes[0].set_title('投入 vs 力学エネルギー')
axes[0].legend()

# --- 平均速度 ---
axes[1].bar(x, runs_df['v_avg_m_s'], color='seagreen')
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, fontsize=8)
axes[1].set_ylabel('平均速度 [m/s]')
axes[1].set_title('走行の平均速度')

# --- エネルギー効率 ---
axes[2].bar(x, runs_df['efficiency_pct'], color='tomato')
axes[2].set_xticks(x)
axes[2].set_xticklabels(labels, fontsize=8)
axes[2].set_ylabel('効率 [%]')
axes[2].set_title('エネルギー効率（力学E / 投入E）')

plt.tight_layout()
plt.savefig('../data/energy_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print('グラフ保存完了 → data/energy_summary.png')

## 5. セクション別タイム分析

各セクションの通過時間からセクション比速度を算出し、  
どのセクションでエネルギーが消費されているかを推定します。

In [ ]:
def parse_section_times(row):
    """section_times_s 列をリストに変換（1周分を抽出）"""
    times = list(map(float, str(row['section_times_s']).split(',')))
    n_sections = len(course_df)
    # 1周分のみ使用
    return times[:n_sections]

# 全走行の平均セクション通過時間
section_times_all = np.array([parse_section_times(row) for _, row in runs_df.iterrows()])
mean_section_times = section_times_all.mean(axis=0)

# セクション速度
section_lengths_m = course_df['length_mm'].values / 1000
mean_section_speeds = section_lengths_m / mean_section_times  # [m/s]

# セクションごとの運動エネルギー差分（近似）
avg_mass_kg = runs_df['car_weight_kg'].mean()
section_KE = 0.5 * avg_mass_kg * mean_section_speeds ** 2

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
fig.suptitle('セクション別 速度・エネルギー分析', fontsize=13)

x_sec = np.arange(len(course_df))
sec_labels = [f"S{i+1}\n{t}" for i, t in enumerate(course_df['type'])]

axes[0].bar(x_sec, mean_section_speeds, color='royalblue')
axes[0].set_xticks(x_sec)
axes[0].set_xticklabels(sec_labels, fontsize=7)
axes[0].set_ylabel('推定区間速度 [m/s]')
axes[0].set_title('セクション別 平均速度')

axes[1].bar(x_sec, section_KE, color='coral')
axes[1].set_xticks(x_sec)
axes[1].set_xticklabels(sec_labels, fontsize=7)
axes[1].set_ylabel('運動エネルギー [J]')
axes[1].set_title('セクション別 運動エネルギー（½mv²）')

plt.tight_layout()
plt.savefig('../data/section_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('グラフ保存完了 → data/section_analysis.png')

## まとめ

| 項目 | 内容 |
|---|---|
| コース1周距離 | `total_length_m` m |
| 投入エネルギー | 電圧降下 × 容量 × 3600 [J] |
| 走行力学エネルギー | ½mv²（最高速推定）+ mgh（スロープ最高点）|
| エネルギー効率 | 力学E ÷ 投入E × 100 [%] |

### 改善のヒント
- **効率が低い** → ギア・タイヤの見直し、モーターの慣らし
- **スロープセクションが遅い** → スプリング・バンパーの調整
- **電池の電圧降下が大きい** → 内部抵抗の小さい電池に変更